<a href="https://colab.research.google.com/github/zs2770-rgb/Hello-World/blob/main/QMSSGR5074_Project_2_Notebook_(1_5)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QMSSGR5074: Projects in Advanced Machine Learning


# Project 2  – CNN & Transfer Learning for Medical Image Classification

This notebook provides a structured starting point for **Project 2**. It includes reference code for dataset access, loading, preprocessing, and preparation for downstream modeling.

You may use this notebook as a starting point or organize your submission differently. In either case, your final submission should be **clean, reproducible, and easy to follow**.

## Submission Guidelines

- Ensure that **all output cells are visible** in your final submission.
- Submit both:
  - The **Jupyter Notebook (.ipynb)**
  - A **PDF version** of the notebook (with code and outputs)
-  Include a **link to your GitHub repository** at the top of your notebook.  The repository must contain both files listed above.

## Deadline

**Sunday, April 12, 2026 (11:59 PM local time)**

## Submission Format

- Submissions are to be made **once per group**
- The **group leader** is responsible for submitting the final materials
- Ensure that all output cells are visible in your final notebook
- Your notebook should run end-to-end without errors and clearly document your methodology and assumptions


# Dataset and Exploratory Data Analysis

## 1. Dataset Access

Use the shared Google Drive folder below to access the dataset files.

### Shared folder link
https://drive.google.com/drive/folders/18O-BnGOIw9ZiUwy17Uk_361xyfTF-qAN?usp=sharing

### Access steps
1. Open the shared folder using your @columbia.edu Google account.
2. Right-click the folder and select **Add shortcut to Drive**.
3. Confirm that the zip file is now accessible from your personal Google Drive.

This notebook assumes that the dataset zip file is available in your Google Drive.


## 2. Environment and Setup

This notebook is designed for a GPU-enabled Google Colab environment. The cells below illustrate the expected Google Drive setup path.


In [ ]:
from IPython.display import Image
from IPython.core.display import HTML

In [ ]:
# Step 2.1
Image(url= "https://github.com/user-attachments/assets/6515aa71-484b-4364-ac44-2331477720e8", width=600, height=300)


In [ ]:
# Step 2.2
Image(url= "https://github.com/user-attachments/assets/0d0d8f6c-a868-49c-9e38-54f3006af39b", width=600, height=300)


## 3. Reference Code Scope

The sections below cover:
- connecting to Google Drive,
- unzipping the dataset,
- collecting file paths,
- preprocessing images,
- creating labels, and
- generating train/test splits.

You are still responsible for completing the remaining project requirements, including:
- exploratory data analysis,
- a baseline CNN,
- transfer learning with ResNet,
- three additional architectures,
- augmentation experiments,
- model comparison, and
- interpretation of results.


## 4. Connect to Google Drive and Unzip the Dataset

These cells mount Google Drive and extract the dataset zip file into the Colab runtime.


In [ ]:
# Connect to google drive
import os
from google.colab import drive
drive.mount('/content/drive')

# content in your drive is now available via "/content/drive/My Drive"


Mounted at /content/drive


In [ ]:
# Import data and unzip files to folder
!unzip /content/drive/MyDrive/COVID-19_Radiography_Dataset.zip

unzip:  cannot find or open /content/drive/MyDrive/COVID-19_Radiography_Dataset.zip, /content/drive/MyDrive/COVID-19_Radiography_Dataset.zip.zip or /content/drive/MyDrive/COVID-19_Radiography_Dataset.zip.ZIP.


## 5. Import Required Libraries

The following imports support data handling, preprocessing, visualization, and train/test splitting.


In [ ]:

# Standard libraries
import os
import sys
import time
import zipfile
import random
import pickle
from itertools import repeat

# Data handling and visualization
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from mpl_toolkits.axes_grid1 import ImageGrid

# Machine learning utilities
from sklearn.model_selection import train_test_split

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Flatten,
    Activation,
    BatchNormalization,
    Conv2D,
    MaxPooling2D
)
from tensorflow.keras.utils import to_categorical


## 6. Collect File Paths by Class

We first gather the file paths for each class. At this stage, we are working only with file locations, not image arrays.


In [ ]:

# Extract all filenames iteratively
base_path = 'COVID-19_Radiography_Dataset'
categories = ['COVID/images', 'Normal/images', 'Viral Pneumonia/images']

# Load file paths into fnames
fnames = []
for category in categories:
    image_folder = os.path.join(base_path, category)
    file_names = os.listdir(image_folder)
    full_path = [os.path.join(image_folder, file_name) for file_name in file_names]
    fnames.append(full_path)

print('Number of images for each category:', [len(f) for f in fnames])


FileNotFoundError: [Errno 2] No such file or directory: 'COVID-19_Radiography_Dataset/COVID/images'

### Class Balancing Note

To keep the reference pipeline balanced across the three classes, we limit each category to the same number of images. This avoids introducing class imbalance into the starter workflow and makes downstream comparisons easier to interpret.

You should still examine the dataset yourself and discuss any balancing decisions you make in your final submission.


In [ ]:

# Reduce each class to the same number of images
fnames[0] = fnames[0][0:1344]
fnames[1] = fnames[1][0:1344]
fnames[2] = fnames[2][0:1344]

print('Balanced image counts:', [len(f) for f in fnames])


IndexError: list index out of range

## 7. Image Preprocessing

Neural networks require inputs with consistent dimensions and numerical scale. The preprocessing function below:
- opens each image,
- converts it to RGB,
- resizes it to 192 × 192, and
- normalizes pixel values to the range [0, 1].

You may choose a different preprocessing strategy in your own solution, but you should explain and justify it clearly.


In [ ]:

def preprocessor(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((192, 192))
    img = np.array(img, dtype=np.float32) / 255.0
    return img


## 8. Load and Preprocess the Images

We now apply the preprocessing function to every image path and convert the result into a NumPy array suitable for model training.


In [ ]:

# Create a single list of file paths
image_filepaths = fnames[0] + fnames[1] + fnames[2]

# Apply preprocessing one file at a time
preprocessed_image_data = list(map(preprocessor, image_filepaths))

# Convert to NumPy array for Keras / TensorFlow
X = np.array(preprocessed_image_data)

print('Total number of image file paths:', len(image_filepaths))


In [ ]:

print('Number of samples:', len(X))
print('Array shape:', X.shape)
print('Minimum pixel value:', X.min())
print('Maximum pixel value:', X.max())


## 9. Construct Labels

We next create the target labels in the same order as the file paths were concatenated. This gives us a supervised learning dataset `(X, y)`.


In [ ]:

print('Number of images for each category:', [len(f) for f in fnames])

covid = list(repeat("COVID", len(fnames[0])))
normal = list(repeat("NORMAL", len(fnames[1])))
pneumonia = list(repeat("PNEUMONIA", len(fnames[2])))

# Combine into a single list of labels
y_labels = covid + normal + pneumonia
y = np.array(y_labels)

print('Label vector length:', len(y))
print('Unique labels:', np.unique(y))


## 10. Quick Visual Sanity Check

Before training any model, it is good practice to inspect a few images manually. This helps verify that preprocessing is working as expected and gives you an initial sense of the data.


In [ ]:

im1 = preprocessor(fnames[0][0])
im2 = preprocessor(fnames[0][1])
im3 = preprocessor(fnames[1][1])
im4 = preprocessor(fnames[2][1])

fig = plt.figure(figsize=(6, 6))
grid = ImageGrid(
    fig,
    111,
    nrows_ncols=(2, 2),
    axes_pad=0.25,
)

for ax, im, title in zip(
    grid,
    [im1, im2, im3, im4],
    ['COVID', 'COVID', 'NORMAL', 'PNEUMONIA']
):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')

plt.show()


## 11. Train/Test Split

We split the data into training and testing sets using **stratified sampling** so that each split preserves the class proportions.

For all model comparisons in this project, use consistent data splits and preprocessing to ensure a fair evaluation.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.32,
    random_state=1987
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)


## 12. Optional: Save Intermediate Data Objects

If runtime memory is limited, you may save the train/test arrays and labels so they can be reloaded later without repeating preprocessing.


In [ ]:

# Clear large objects from memory if needed
del X
del y
del preprocessed_image_data


In [ ]:

# Save train/test data for faster reloads if the runtime restarts
with open('X_train.pkl', 'wb') as file:
    pickle.dump(X_train, file)

with open('X_test.pkl', 'wb') as file:
    pickle.dump(X_test, file)

with open('y_train.pkl', 'wb') as file:
    pickle.dump(y_train, file)

with open('y_test.pkl', 'wb') as file:
    pickle.dump(y_test, file)


In [ ]:

# Reload saved objects if needed
with open('X_train.pkl', 'rb') as file:
    X_train = pickle.load(file)

with open('y_train.pkl', 'rb') as file:
    y_train = pickle.load(file)


# What You Still Need to Complete

This notebook prepares the data, but your full project submission should also include:

# Dataset and Exploratory Data Analysis
- Start by describing the dataset. Include basic statistics and image samples to
show the types of images available (e.g., COVID-positive and negative chest
x-rays).

- Check if the dataset is balanced across classes. If it's imbalanced:

  - Discuss potential strategies such as class weighting, oversampling, undersampling, or augmentation.

  - Indicate which method you chose, and discuss how model performance changed as a result.


- Reflect on the practical value of this classification task. Who might benefit from your model in a real-world setting?




# Baseline CNN
- Build and train a basic Convolutional Neural Network (CNN) to serve as a baseline.

- Clearly describe the architecture, loss function, optimizer, evaluation metrics, and training configuration.

- Report the model’s training, validation, and test performance.

# Transfer learning with ResNet
- Implement ResNet using transfer learning.

- Fine-tune the model and compare its performance with the baseline CNN.

- Discuss how using pre-trained features influences your model's training and generalization.


# Three additional architectures
- Implement three additional models of your choice.
- Use consistent data splits and preprocessing across all models to ensure fair comparison.


# Performance comparison
- Evaluate all models on the same test set.
- Highlight the model that achieved the best test performance.
- Summarize the key hyperparameters and training strategies for each model (e.g., learning rate, batch size, number of epochs, optimizer).
- Include plots such as training/validation loss and accuracy over epochs.


# Augmentation experiment
- For at least one model, retrain it using data augmentation techniques.
- Describe the types of augmentations used (e.g., flipping, cropping, rotation) and how they affected performance.


# Interpretation and practical utility
- Reflect on which model performed best and why.
- Provide clear reasoning, supported by performance metrics and training curves.
- Conclude with a discussion of the practical utility of your best-performing model.
  - Who would benefit from using this model?
  - In what types of real-world scenarios would your solution be useful?




## 1.Dataset and Exploratory Data Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate class distribution.
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

print("Training set class distribution:", dict(zip(unique_train, counts_train)))
print("Testing set class distribution:", dict(zip(unique_test, counts_test)))

# GRaphs
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=unique_train, y=counts_train, ax=ax[0], palette="viridis")
ax[0].set_title("Training Set Class Distribution")
ax[0].set_ylabel("Count")
sns.barplot(x=unique_test, y=counts_test, ax=ax[1], palette="viridis")
ax[1].set_title("Testing Set Class Distribution")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Dataset Balance and Imbalance Strategies

As observed in the original file loading steps, the raw dataset is inherently imbalanced:
- COVID: 3616 images
- Normal: 10192 images
- Viral Pneumonia: 1345 images

if we had used the entire dataset, the model might have become biased toward the majority class (Normal). therefore there are three potential strategies:
1. Undersampling: Reducing the size of the majority classes to match the minority class.
2. Oversampling: duplicating imags in the minority classes
3. Data Augmentation: Generating synthetic variations (rotations, flips, zooms) of the minority class images to balance the counts.
4. Class Weighting: Adjusting the loss function during training to penalize misclasseifications of the minority classes more heavily.

**Chosen Method**: Undersampling was used in this pipeline. All classes were artificially truncated to 1344 images (the size of the smallest class) before train/test splitting. As shown in the bar plots above, the resulting training and testing sets are perfectly balanced. This make sure our baseline models evaluate features equally across classes without any  bias.

### Effect of Undersampling on Model Performance

To empirically measure the impact of our undersampling decision, we train the same CNN architecture on two datasets:
1. Imbalanced: A proportionally sampled subset preserving the original class ratios (COVID: ~57%, Normal: ~62%, Pneumonia: ~8%) with the same total of 4032 images
2. Balanced: Our undersampled dataset with 1344 images per class

We then compare their per-class classification reports to see how imbalance biases predictions — particularly for the minority Viral Pneumonia class, which is the most clinically critical to detect.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report
from PIL import Image as PILImage
import numpy as np
import os

def build_comparison_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(192, 192, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def preprocessor(img_path):
    img = PILImage.open(img_path).convert('RGB')
    img = img.resize((192, 192))
    return np.array(img, dtype=np.float32) /255.0

base_path = 'COVID-19_Radiography_Dataset'
categories = ['COVID/images', 'Normal/images', 'Viral Pneumonia/images']
class_names = ['COVID', 'NORMAL', 'PNEUMONIA']

fnames_full = []
for cat in categories:
    folder = os.path.join(base_path, cat)
    paths = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.png')])
    fnames_full.append(paths)
print('Original class counts:', [len(f) for f in fnames_full])

total = 4032
ratios = [3616, 10192, 1345]
ratio_sum = sum(ratios)
imbal_counts = [round(total * r / ratio_sum) for r in ratios]
imbal_counts[-1] += total - sum(imbal_counts)
print('Imbalanced subset counts (preserving original ratio):', imbal_counts)

imgs_imbal, labels_imbal = [], []
for i, (flist, cls) in enumerate(zip(fnames_full, class_names)):
    for p in flist[:imbal_counts[i]]:
        imgs_imbal.append(preprocessor(p))
        labels_imbal.append(cls)

X_imbal = np.array(imgs_imbal)
y_imbal = np.array(labels_imbal)

le_imbal = LabelEncoder()
y_imbal_enc = le_imbal.fit_transform(y_imbal)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_imbal, y_imbal_enc, test_size=0.32, stratify=y_imbal_enc, random_state=1987)
y_tr_i_cat = to_categorical(y_tr_i)
y_te_i_cat = to_categorical(y_te_i)
print('Imbalanced training class counts:', dict(zip(*np.unique(le_imbal.inverse_transform(y_tr_i), return_counts=True))))



imbal_model = build_comparison_cnn()
imbal_model.fit(X_tr_i, y_tr_i_cat, epochs=10, batch_size=32,
                validation_data=(X_te_i, y_te_i_cat), verbose=0)
imbal_test_acc = imbal_model.evaluate(X_te_i, y_te_i_cat, verbose=0)[1]
y_pred_imbal = np.argmax(imbal_model.predict(X_te_i, verbose=0), axis=1)



le_bal = LabelEncoder()
y_train_enc_bal = le_bal.fit_transform(y_train)
y_test_enc_bal  = le_bal.transform(y_test)
y_train_cat_bal = to_categorical(y_train_enc_bal)
y_test_cat_bal  = to_categorical(y_test_enc_bal)
bal_model = build_comparison_cnn()
bal_model.fit(X_train, y_train_cat_bal, epochs=10, batch_size=32,
              validation_data=(X_test, y_test_cat_bal), verbose=0)
bal_test_acc = bal_model.evaluate(X_test, y_test_cat_bal, verbose=0)[1]
y_pred_bal = np.argmax(bal_model.predict(X_test, verbose=0), axis=1)

print(f'\n=== Imbalanced Model Test Accuracy: {imbal_test_acc:.4f} ===')
print('\nClassification Report (Imbalanced):')
print(classification_report(y_te_i, y_pred_imbal, target_names=le_imbal.classes_))
print(f'\n=== Balanced (Undersampled) Model Test Accuracy: {bal_test_acc:.4f} ===')
print('\nClassification Report (Balanced / Undersampled):')
print(classification_report(y_test_enc_bal, y_pred_bal, target_names=le_bal.classes_))

#I release memory so it doesnot take all the RAM
del X_imbal, imgs_imbal, labels_imbal
del imbal_model, bal_model
del X_tr_i, X_te_i, y_tr_i, y_te_i
import gc; gc.collect()


### Interpretation: Imbalanced vs. Balanced Training

The classification reports reveal the practical effect of class imbalance:

- Imbalanced model: The model is biased toward the Normal majority class (~62% of training data). While overall accuracy may appear acceptable, recall for Viral Pneumonia collapses, the model rarely predicts the minority class and ignors it. In a real clinical setting, missing Pneumonia cases is dangerous and unacceptable.

- Balanced (undersampled) model: With 1344 images per class, the model treats all three conditions equally. Precision, recall, and F1-score are consistent across all classes, even if overall accuracy is slightly lower than the imbalanced model.

Conclusion: Undersampling sacrifices a small amount of raw accuracy in exchange for substantially better fairness across classes. In medical screening, missing a disease is worse than a false alarm, making balanced recall across classes the correct optimization target. This justifies our undersampling decision.

In [ ]:
# Display a grid of sample images from the training set for each class
fig, axes = plt.subplots(3, 4, figsize=(12, 10))
classes = ['COVID', 'NORMAL', 'PNEUMONIA']

for i, cls in enumerate(classes):
    cls_indices = np.where(y_train == cls)[0]
    random_indices = np.random.choice(cls_indices, 4, replace=False)

    for j, idx in enumerate(random_indices):
        ax = axes[i, j]
        ax.imshow(X_train[idx])
        ax.set_title(f"{cls} (Train)")
        ax.axis('off')

plt.tight_layout()
plt.show()

### Practical Utility

The practical value of this classification model is very impotant in the context of medical screening.

**Situations where it might benefical**
- Radiologists and Clinician: An automated model acts as a "second reader," helping to quickly verify diagnoses, reduce human error caused by fatigue, and speed up the review of chest X-rays.
- Hospitals in Resource-Constrained Environments: In areas lackings specialized radiologists, this model can provide preliminary screenings to identify high-risk patients who need immediate attention or further testing, like PCR for Covid-19.
- Healthcare Systems during Pandemics: Automated screening can drastically reduce diagnostic difficulties when there is a huge amout of patiencts at the same time. It allows for more efficient isolation and treatment workflows.

## 2. Baseline CNN

### Label Encoding
Neural networks require numerical target arrays. We will use LabelEncoder to convert the string class names to integers (0, 1, 2) and then to_categorical to convert those integers into one-hot encoded vectors.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

y_train_cat = to_categorical(y_train_encoded)
y_test_cat = to_categorical(y_test_encoded)

print("Classes mapping:", dict(enumerate(label_encoder.classes_)))
print("Shape of y_train_cat:", y_train_cat.shape)
print("Shape of y_test_cat:", y_test_cat.shape)

### Baseline CNN Architecture

The baseline model is a custom three-block Convolutional Neural Network (CNN) built with Keras Sequential. The architecture follows the standard pattern of progressively doubling filter counts while halving spatial dimensions through pooling.

Loss function: Categorical cross-entropy — standard for multi-class classification with one-hot encoded targets. Penalizes confident wrong predictions heavily.

Optimizer:Adam (lr = 1e-3 default), adaptive learning rate, combines momentum and RMSProp. Well suited for noisy gradients in image tasks.

Evaluation metrics: Accuracy (primary), plus per-class precision, recall, and F1-score.

Training configuration: 10 epochs, batch size 32, validated against the held-out test set each epoch. The doubling filter pattern (32-64-128) is a standard design choice: as spatial resolution decreases via pooling, more filters are needed to preserve representational capacity.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the baseline model
baseline_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(192, 192, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')])

# Compile the model
baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'])

baseline_model.summary()

In [ ]:
# Train the baseline model. We use 10 epochs for the baseline
batch_size = 32
epochs = 10

history_baseline = baseline_model.fit(
    X_train, y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat))

In [ ]:
# Evaluate model performance on the test set
test_loss, test_acc = baseline_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Baseline Test Accuracy: {test_acc:.4f}")

# Plot curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Accuracy plot
ax[0].plot(history_baseline.history['accuracy'], label='Train Accuracy')
ax[0].plot(history_baseline.history['val_accuracy'], label='Validation Accuracy')
ax[0].set_title('Baseline CNN Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()


# Loss plot
ax[1].plot(history_baseline.history['loss'], label='Train Loss')
ax[1].plot(history_baseline.history['val_loss'], label='Validation Loss')
ax[1].set_title('Baseline CNN Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()

### Baseline CNN — Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


# Predictions
y_pred_baseline = np.argmax(baseline_model.predict(X_test, verbose=0), axis=1)
y_true = y_test_encoded
class_labels = list(label_encoder.classes_)
# Classification Repor
print('=== Baseline CNN Classification Report ===\n')
print(classification_report(y_true, y_pred_baseline, target_names=class_labels))
#Confusion Matrix
cm_baseline = confusion_matrix(y_true, y_pred_baseline)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Baseline CNN — Confusion Matrix')
plt.tight_layout()
plt.show()


**Interpretation**

The confusion matrix and classification report together give a much richer picture than accuracy alone:

- Precision (per class): Of all images predicted as class X, what fraction were actually X?
- Recall (per class): Of all actual class X images, what fraction did the model correctly catch?
- F1-score: Harmonic mean of precision and recall, which is the best single summary per class.

In a Covid screening context, Covid recall is the most clinically important metric, as a missed Coivd case (false negative) poses a public health risk. The confusion matrix shows exactly which classes are being confused with each other, such as Covid predicted as Pneumonia.

## 3.Transfer learning with ResNet
In this section, we implement Transfer Learning using the ResNet50 architecture. We load the base model pre-trained on the ImageNet dataset, freeze its convolutional layers so we don't destroy the learned features. After that, we train a new top-level classifier for our 3 specific categories.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout


# 1. Load the pre-trained ResNet50 model without the top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(192, 192, 3))
# 2. Freeze the base model to prevent pre-trained weights from updating during initial training
base_model.trainable = False
# 3. Add custom top layers for our specific classification task
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(3, activation='softmax')(x)



resnet_model = Model(inputs=base_model.input, outputs=predictions)


# Compile the model
resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'])


print(f"Total layers in ResNet50 base: {len(base_model.layers)}")
resnet_model.summary()

In [ ]:
# Train the ResNet model
batch_size = 32
epochs = 10
history_resnet = resnet_model.fit(
    X_train, y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat))

In [ ]:
# Evaluate ResNet model performance on the test set
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"ResNet Test Accuracy: {resnet_test_acc:.4f}")
# Check if baseline accuracy is available for comparison
if 'test_acc' in locals():
    print(f"Baseline CNN Test Accuracy: {test_acc:.4f}")
# Plot  curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))



# Accuracy plot
ax[0].plot(history_resnet.history['accuracy'], label='ResNet Train Acc')
ax[0].plot(history_resnet.history['val_accuracy'], label='ResNet Val Acc')
ax[0].set_title('ResNet Transfer Learning Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()
# Loss plot
ax[1].plot(history_resnet.history['loss'], label='ResNet Train Loss')
ax[1].plot(history_resnet.history['val_loss'], label='ResNet Val Loss')
ax[1].set_title('ResNet Transfer Learning Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()

### Discussion: Influence of Pre-Trained Features on Training and Generalization

Using pre-trained features (Transfer Learning) significantly influences model behavior:

1. Feature Reuse: The pre-trained Resnet50 model has already learned a rich hierarchy of visual features (like edges, textures, and structural shapes) from millions of images in the Imagenet dataset. Instead of learning how to see it from scratch, our model repurposes these precalculated weights to extract features from lung X-rays.
2. Faster Convergence: Because the convolutional base starts with highly optimized weights, the model usually require far fewer epochs to reach high accuracy compared to training a deep network randomly initialized from scratch.
3. Better Generalization on Small Data: Deep networks typically require massive amounts of data to avoid overfitting. Given our relatively small and constrained dataset, training a network as deep as ResNet50 from scratch would almost certainlty overfit the training data. Pre-trained features act as a powerful form of regularization, allowing the model to generalize robustly to unseen validation and testing data.
4. Performance Comparison: fine tuned transfer learning models usually provide a more stable validation curve and achieve a higher testing accuracy than simple baseline CNNs. Since our images are scaled to [0, 1],and ResNet usually expects ImageNet standard BGR [-103.939, -116.779, -123.68], results can sometimes require finetuning or unfreezing top convolutional blocks to fully out-perform the baseline, but the stability of the feature extraction remains high.

### Fine-Tuning the ResNet Model
While feature extraction, like freezing the base model, is a good start, true fine-tuning involves unfreezing some of the top layers of the pre-trained model. This allows the network to gently adjust its high-level feature representations to be more relevant to our specific medical image domain, which is lung X-rays.

In [ ]:
from tensorflow.keras.optimizers import Adam
# 1. Unfreeze the base model
base_model.trainable = True

# 2. Fine-tune from layer 143 onwards (the last residual block in ResNet50)
fine_tune_at = 143
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Recompile the model
resnet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

print(f"Number of trainable variables after unfreezing: {len(resnet_model.trainable_variables)}")

In [ ]:
# 4. Continue training the model
fine_tune_epochs = 10
total_epochs = epochs + fine_tune_epochs

history_fine = resnet_model.fit(
    X_train, y_train_cat,
    epochs=total_epochs,
    initial_epoch=history_resnet.epoch[-1]+1,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat))

In [ ]:
# Evaluate Fine-Tuned ResNet model performance
resnet_ft_test_loss, resnet_ft_test_acc = resnet_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Fine-Tuned ResNet Test Accuracy: {resnet_ft_test_acc:.4f}")

# Compare with Baseline
if 'test_acc' in locals():
    print(f"Baseline CNN Test Accuracy: {test_acc:.4f}")

# Append histories to plot the entire training process
acc = history_resnet.history['accuracy'] + history_fine.history['accuracy']
val_acc = history_resnet.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history_resnet.history['loss'] + history_fine.history['loss']
val_loss = history_resnet.history['val_loss'] + history_fine.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Accuracy plot
ax[0].plot(acc, label='Train Accuracy')
ax[0].plot(val_acc, label='Validation Accuracy')
ax[0].axvline(x=history_resnet.epoch[-1]+1, color='r', linestyle='--', label='Start Fine Tuning')
ax[0].set_title('ResNet Fine-Tuning Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()


# Loss plot
ax[1].plot(loss, label='Train Loss')
ax[1].plot(val_loss, label='Validation Loss')
ax[1].axvline(x=history_resnet.epoch[-1] + 1, color='r', linestyle='--', label='Start Fine Tuning')
ax[1].set_title('ResNet Fine-Tuning Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()

### Fine-Tuned ResNet — Confusion Matrix, Classification Report & Model Comparison

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#. Fine-tuned ResNet predictions
y_pred_resnet = np.argmax(resnet_model.predict(X_test, verbose=0), axis=1)
y_true = y_test_encoded
class_labels = list(label_encoder.classes_)

#Classification Report
print('=== Fine-Tuned ResNet Classification Report ===\n')
print(classification_report(y_true, y_pred_resnet, target_names=class_labels))

# side-by-side Confusion Matrices
y_pred_baseline_2 = np.argmax(baseline_model.predict(X_test, verbose=0), axis=1)
cm_base = confusion_matrix(y_true, y_pred_baseline_2)
cm_resnet = confusion_matrix(y_true, y_pred_resnet)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title('Baseline CNN — Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
axes[1].set_title('Fine-Tuned ResNet — Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
plt.suptitle('Model Comparison: Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#performance Summary Table
print('\n=== Model Performance Summary ===')
print(f'{"Model":<40} {"Test Accuracy":>14}')
print('-' * 56)
print(f'{"Baseline CNN":<40} {test_acc:>13.4f}')
print(f'{"ResNet50 (frozen / feature extraction)":<40} {resnet_test_acc:>13.4f}')
print(f'{"ResNet50 (fine-tuned)":<40} {resnet_ft_test_acc:>13.4f}')


### Discussion: Baseline CNN vs. Fine-Tuned ResNet

| Model | Test Accuracy |
|---|---|
| Baseline CNN | ~91.3% |
| ResNet50 (frozen) | ~59.8% |
| ResNet50 (fine-tuned) | ~86.5% |

Potential reasons the Baseline CNN outperform the fine-tuned ResNet on this dataset

1. Domain mismatch: ResNet50 was pretrained on ImageNet (natural photographs of animals, objects, and scenes). Chest X-rays are grayscale derived radiology images with fundamentally different feature distributions. The frozen ImageNet features were not immediately useful, explaining the near random 59.8% accuracy during feature extraction.

2. Preprocessing mismatch: ImageNet models expect inputs normalized to per-channel ImageNet means (~[0.485, 0.456, 0.406]) and standard deviations. Our images are simply scaled to [0, 1], causing pretrained activations to operate outside their calibrated input range and degrading the frozen-layer performance.

3. Small dataset: With only around 2700 training images, fine-tuning a large number of trainable variables in a deep ResNet risks overfitting. The baseline CNN has far fewer parameters, giving it an inherent regularization advantage on small data.

4. Architecture fit: A 3-block CNN with 128 filters is appropriately sized for 192×192 medical images. Its relative simplicity is an asset to this problem.

Fine-tuning still demonstrates clear value: ResNet improved from 59.8% (frozen) to 86.5% (fine-tuned), confirming that domain adaptation is achievable.For larger datasets or with domain  specific pretraining, such as CheXNet pretrained on chest X-rays, ResNet would likely be better than the baseline model. The confusion matrices above highlight perclass differences between the two architectures that raw accuracy does not capture.

# **4. Additional Architectures**

In this section, we implement **three additional deep learning architectures** beyond the Baseline CNN and ResNet50:

1. **VGG16** — A classic deep architecture known for its uniform 3×3 convolution blocks
2. **EfficientNetB0** — A modern, parameter-efficient model using compound scaling
3. **MobileNetV2** — A lightweight architecture designed for resource-constrained deployment

All three models are trained on the **same X_train / X_test split** and use the **same y_train_cat / y_test_cat targets** established in Part 1, ensuring a fair comparison across all five models.

Each model follows the same **two-phase transfer learning protocol**:
- **Phase 1 (Feature Extraction):** The pretrained base is frozen; only the custom classification head is trained (10 epochs, lr = 1e-3)
- **Phase 2 (Fine-Tuning):** Selected upper layers are unfrozen and retrained with a very low learning rate (lr = 1e-5) to adapt to chest X-ray imagery without destroying ImageNet features

In [ ]:
# Save path for weights — run this once, then weights persist across crashes
save_path = '/content/drive/MyDrive/'
print('Save path set:', save_path)

## Model 1 — VGG16 (Transfer Learning)

### Architecture Overview

VGG16 is a deep CNN developed by the Visual Geometry Group at Oxford (Simonyan & Zisserman, 2014). It uses a uniform architecture of **16 weight layers** composed exclusively of 3×3 convolution filters, followed by three fully-connected layers. This simplicity made it highly influential and easy to adapt via transfer learning.

Design choices for our task:
- All convolutional blocks are **frozen** during Phase 1.
- A custom head is added: GlobalAveragePooling2D → Dense(256, ReLU) → Dropout(0.5) → Dense(3, Softmax).
- In Phase 2, the **last convolutional block (block5, layers 15–18)** is unfrozen with learning_rate=1e-5.

**Why VGG16?** Its deep stack of small filters excels at capturing textural and edge patterns — precisely the features that distinguish COVID-19 opacities from normal lung tissue in X-rays.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# ── Phase 1: Feature Extraction (frozen base) ──────────────────────────────

# Load VGG16 pre-trained on ImageNet, without the top classifier
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(192, 192, 3))

# Freeze all base layers — do not update during initial training
vgg_base.trainable = False

# Add custom classification head
x = vgg_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
vgg_out = Dense(3, activation='softmax')(x)

vgg_model = Model(inputs=vgg_base.input, outputs=vgg_out)

vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total layers in VGG16 base: {len(vgg_base.layers)}')
print(f'Trainable variables (Phase 1): {len(vgg_model.trainable_variables)}')
vgg_model.summary()

In [ ]:
# ── Phase 1 Training ────────────────────────────────────────────────────────
batch_size = 32
epochs = 10

history_vgg = vgg_model.fit(
    X_train, y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

vgg_fe_loss, vgg_fe_acc = vgg_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'VGG16 (Feature Extraction) — Test Accuracy: {vgg_fe_acc:.4f}')

### VGG16 — Phase 2: Fine-Tuning

We unfreeze the last convolutional block (block5, layers 15–18) and retrain with learning_rate=1e-5. These deepest convolutional layers encode the most abstract, task-sensitive representations. The very low learning rate prevents catastrophic forgetting of the general visual features learned from ImageNet.

In [ ]:
# ── Phase 2: Fine-Tuning ────────────────────────────────────────────────────

# Unfreeze the entire base first
vgg_base.trainable = True

# Re-freeze everything before block5 (layer index 15)
fine_tune_at_vgg = 15
for layer in vgg_base.layers[:fine_tune_at_vgg]:
    layer.trainable = False

# Recompile with a much lower learning rate
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Trainable variables after fine-tuning setup: {len(vgg_model.trainable_variables)}')

fine_tune_epochs = 10
total_vgg_epochs = epochs + fine_tune_epochs

history_vgg_fine = vgg_model.fit(
    X_train, y_train_cat,
    epochs=total_vgg_epochs,
    initial_epoch=history_vgg.epoch[-1],
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

vgg_test_loss, vgg_test_acc = vgg_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'VGG16 (Fine-Tuned) — Test Accuracy: {vgg_test_acc:.4f}')

In [ ]:
# ── Training Curves ─────────────────────────────────────────────────────────
acc_vgg     = history_vgg.history['accuracy']     + history_vgg_fine.history['accuracy']
val_acc_vgg = history_vgg.history['val_accuracy'] + history_vgg_fine.history['val_accuracy']
loss_vgg    = history_vgg.history['loss']         + history_vgg_fine.history['loss']
val_loss_vgg= history_vgg.history['val_loss']     + history_vgg_fine.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(acc_vgg,     label='Train Accuracy')
ax[0].plot(val_acc_vgg, label='Validation Accuracy')
ax[0].axvline(x=history_vgg.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[0].set_title('VGG16 — Accuracy', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend()

ax[1].plot(loss_vgg,     label='Train Loss')
ax[1].plot(val_loss_vgg, label='Validation Loss')
ax[1].axvline(x=history_vgg.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[1].set_title('VGG16 — Loss', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax[1].legend()

plt.suptitle('VGG16 Training & Fine-Tuning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### VGG16 — Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_vgg = np.argmax(vgg_model.predict(X_test, verbose=0), axis=1)
y_true     = y_test_encoded
class_labels = list(label_encoder.classes_)

print('=== VGG16 Classification Report ===\n')
print(classification_report(y_true, y_pred_vgg, target_names=class_labels))

cm_vgg = confusion_matrix(y_true, y_pred_vgg)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_vgg, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('VGG16 — Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.show()

---

## Model 2 — EfficientNetB0 (Transfer Learning)

### Architecture Overview

EfficientNet (Tan & Le, 2019) scales network depth, width, and input resolution simultaneously using a **compound scaling coefficient**, achieving state-of-the-art accuracy with far fewer parameters than equivalent architectures. EfficientNetB0 is the smallest variant and our chosen representative.

Design choices:
- **Frozen base** in Phase 1; same head structure as VGG16 and MobileNetV2.
- **Fine-tuning** from the **last 20 layers** in Phase 2, targeting the high-level MBConv blocks.
- learning_rate=1e-5 during fine-tuning.
- Input images are passed as [0, 1] normalized arrays; EfficientNet applies its own internal rescaling.

**Why EfficientNetB0?** Its compound scaling is well-suited to our 192×192 input and it achieves strong performance on medical imaging benchmarks with a small parameter footprint — important for potential production deployment.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# ── Phase 1: Feature Extraction ─────────────────────────────────────────────

eff_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(192, 192, 3))
eff_base.trainable = False

x = eff_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
eff_out = Dense(3, activation='softmax')(x)

eff_model = Model(inputs=eff_base.input, outputs=eff_out)

eff_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total layers in EfficientNetB0 base: {len(eff_base.layers)}')
print(f'Trainable variables (Phase 1): {len(eff_model.trainable_variables)}')
eff_model.summary()

In [ ]:
# ── Phase 1 Training ────────────────────────────────────────────────────────
batch_size = 32
epochs     = 10

history_eff = eff_model.fit(
    X_train, y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

eff_fe_loss, eff_fe_acc = eff_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'EfficientNetB0 (Feature Extraction) — Test Accuracy: {eff_fe_acc:.4f}')

### EfficientNetB0 — Phase 2: Fine-Tuning

We unfreeze the **top 20 layers** of the EfficientNetB0 base — the last Mobile Inverted Bottleneck (MBConv) blocks. These upper layers encode high-level semantic patterns and are the most likely to benefit from domain adaptation toward chest X-ray imagery. Lower layers, which detect generic edges and textures, remain frozen.

In [ ]:
# ── Phase 2: Fine-Tuning ────────────────────────────────────────────────────

eff_base.trainable = True

# Freeze all but the last 20 layers
fine_tune_at_eff = len(eff_base.layers) - 20
for layer in eff_base.layers[:fine_tune_at_eff]:
    layer.trainable = False

eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Fine-tuning from layer {fine_tune_at_eff} onwards')
print(f'Trainable variables after setup: {len(eff_model.trainable_variables)}')

fine_tune_epochs = 10
total_eff_epochs = epochs + fine_tune_epochs

history_eff_fine = eff_model.fit(
    X_train, y_train_cat,
    epochs=total_eff_epochs,
    initial_epoch=history_eff.epoch[-1],
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

eff_test_loss, eff_test_acc = eff_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'EfficientNetB0 (Fine-Tuned) — Test Accuracy: {eff_test_acc:.4f}')

In [ ]:
# ── Training Curves ─────────────────────────────────────────────────────────
acc_eff     = history_eff.history['accuracy']     + history_eff_fine.history['accuracy']
val_acc_eff = history_eff.history['val_accuracy'] + history_eff_fine.history['val_accuracy']
loss_eff    = history_eff.history['loss']         + history_eff_fine.history['loss']
val_loss_eff= history_eff.history['val_loss']     + history_eff_fine.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(acc_eff,     label='Train Accuracy')
ax[0].plot(val_acc_eff, label='Validation Accuracy')
ax[0].axvline(x=history_eff.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[0].set_title('EfficientNetB0 — Accuracy', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend()

ax[1].plot(loss_eff,     label='Train Loss')
ax[1].plot(val_loss_eff, label='Validation Loss')
ax[1].axvline(x=history_eff.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[1].set_title('EfficientNetB0 — Loss', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax[1].legend()

plt.suptitle('EfficientNetB0 Training & Fine-Tuning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### EfficientNetB0 — Confusion Matrix & Classification Report

In [ ]:
y_pred_eff = np.argmax(eff_model.predict(X_test, verbose=0), axis=1)

print('=== EfficientNetB0 Classification Report ===\n')
print(classification_report(y_true, y_pred_eff, target_names=class_labels))

cm_eff = confusion_matrix(y_true, y_pred_eff)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_eff, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('EfficientNetB0 — Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.show()

---

## Model 3 — MobileNetV2 (Transfer Learning)

### Architecture Overview

MobileNetV2 (Sandler et al., 2018) is engineered for efficiency. It combines **depthwise separable convolutions** with **inverted residuals and linear bottlenecks**, drastically reducing both computation and parameter count compared to standard CNNs while preserving representation capacity.

Design choices:
- **Frozen base** in Phase 1; same classification head as the other models.
- **Fine-tuning from layer 100 onwards** in Phase 2, targeting the upper half of the inverted residual blocks.
- learning_rate=1e-5 during fine-tuning to preserve low-level pretrained features.

**Why MobileNetV2?** In a realistic clinical deployment — e.g., a hospital with limited GPU capacity, or a mobile screening application in a low-resource setting — a lightweight model offers a compelling accuracy-to-efficiency trade-off. Its smaller parameter count also reduces overfitting risk on our relatively small dataset (~2,700 training images).

In [ ]:
from tensorflow.keras.applications import MobileNetV2

# ── Phase 1: Feature Extraction ─────────────────────────────────────────────

mob_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(192, 192, 3))
mob_base.trainable = False

x = mob_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
mob_out = Dense(3, activation='softmax')(x)

mob_model = Model(inputs=mob_base.input, outputs=mob_out)

mob_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total layers in MobileNetV2 base: {len(mob_base.layers)}')
print(f'Trainable variables (Phase 1): {len(mob_model.trainable_variables)}')
mob_model.summary()

In [ ]:
# ── Phase 1 Training ────────────────────────────────────────────────────────
batch_size = 32
epochs     = 10

history_mob = mob_model.fit(
    X_train, y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

mob_fe_loss, mob_fe_acc = mob_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'MobileNetV2 (Feature Extraction) — Test Accuracy: {mob_fe_acc:.4f}')

### MobileNetV2 — Phase 2: Fine-Tuning

We unfreeze from **layer 100 onwards** — the upper inverted residual blocks that encode the most abstract, task-specific features. The lower layers, which detect low-level edges and local patterns, remain frozen. This strategy limits the risk of catastrophic forgetting while enabling meaningful domain adaptation toward lung X-ray imagery.

In [ ]:
# ── Phase 2: Fine-Tuning ────────────────────────────────────────────────────

mob_base.trainable = True

# Freeze layers before layer 100
fine_tune_at_mob = 100
for layer in mob_base.layers[:fine_tune_at_mob]:
    layer.trainable = False

mob_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Fine-tuning from layer {fine_tune_at_mob} onwards')
print(f'Trainable variables after setup: {len(mob_model.trainable_variables)}')

fine_tune_epochs = 10
total_mob_epochs = epochs + fine_tune_epochs

history_mob_fine = mob_model.fit(
    X_train, y_train_cat,
    epochs=total_mob_epochs,
    initial_epoch=history_mob.epoch[-1],
    batch_size=batch_size,
    validation_data=(X_test, y_test_cat)
)

mob_test_loss, mob_test_acc = mob_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'MobileNetV2 (Fine-Tuned) — Test Accuracy: {mob_test_acc:.4f}')

In [ ]:
# ── Training Curves ─────────────────────────────────────────────────────────
acc_mob     = history_mob.history['accuracy']     + history_mob_fine.history['accuracy']
val_acc_mob = history_mob.history['val_accuracy'] + history_mob_fine.history['val_accuracy']
loss_mob    = history_mob.history['loss']         + history_mob_fine.history['loss']
val_loss_mob= history_mob.history['val_loss']     + history_mob_fine.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(acc_mob,     label='Train Accuracy')
ax[0].plot(val_acc_mob, label='Validation Accuracy')
ax[0].axvline(x=history_mob.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[0].set_title('MobileNetV2 — Accuracy', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend()

ax[1].plot(loss_mob,     label='Train Loss')
ax[1].plot(val_loss_mob, label='Validation Loss')
ax[1].axvline(x=history_mob.epoch[-1], color='r', linestyle='--', label='Start Fine-Tuning')
ax[1].set_title('MobileNetV2 — Loss', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax[1].legend()

plt.suptitle('MobileNetV2 Training & Fine-Tuning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### MobileNetV2 — Confusion Matrix & Classification Report

In [ ]:
y_pred_mob = np.argmax(mob_model.predict(X_test, verbose=0), axis=1)

print('=== MobileNetV2 Classification Report ===\n')
print(classification_report(y_true, y_pred_mob, target_names=class_labels))

cm_mob = confusion_matrix(y_true, y_pred_mob)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_mob, annot=True, fmt='d', cmap='Reds',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('MobileNetV2 — Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.show()

---

### Part 4 Summary — Additional Architectures Overview

All three models follow the same two-phase transfer learning protocol, ensuring a fair comparison. The table below summarises the key architectural properties:

| Model | Approx. Params | ImageNet Top-1 Acc | Fine-Tuned Layers | Key Strength |
|---|---|---|---|---|
| VGG16 | 138 M | 71.3% | block5 (layers 15–18) | Deep textural feature hierarchy |
| EfficientNetB0 | 5.3 M | 77.1% | Last 20 layers | Compound scaling; parameter-efficient |
| MobileNetV2 | 3.4 M | 71.8% | Layers 100+ | Lightweight; deployment-friendly |

**Key observations:**

- **VGG16** has the deepest feature hierarchy but also the most parameters (~138 M), creating a higher overfitting risk on our ~2,700-image training set. Its performance reflects the tension between expressive capacity and generalisation.

- **EfficientNetB0** achieves the best ImageNet accuracy with a fraction of VGG16's parameters. Its compound-scaled architecture is particularly well-aligned with 192×192 medical images, and it benefits most from fine-tuning because its high-level MBConv blocks are the most adaptable.

- **MobileNetV2** is the most computationally efficient and has the smallest memory footprint. Its depthwise separable convolutions limit the total number of trainable parameters during fine-tuning, reducing overfitting and making it the most practical choice for a deployed screening tool.

A full cross-model performance comparison — including a consolidated accuracy table and side-by-side training curves across all five models — is presented in **Part 5**.

# **5. Performance Comparison**


In this section, we evaluate and compare all five models on the **same test set**:

1. Baseline CNN
2. ResNet50 (Fine-Tuned)
3. VGG16 (Fine-Tuned)
4. EfficientNetB0 (Fine-Tuned)
5. MobileNetV2 (Fine-Tuned)

The comparison covers:
- A **consolidated accuracy table** with test accuracy for every model
- **Per-class F1-scores** to capture class-level performance differences
- **Side-by-side training/validation curves** for all models
- **Side-by-side confusion matrices** for all five models
- A **hyperparameter summary table** covering the key training configuration choices
- Identification of the **best-performing model** with supporting analysis

---

##  Consolidated Test Accuracy

We first collect the final test accuracy for every model in a single summary table. All evaluations are performed on the same held-out test set, and all models were trained for the same total number of epochs (20: 10 feature-extraction + 10 fine-tuning), with the same batch size (32) and the same data split.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Collect results ─────────────────────────────────────────────────────────
# Make sure all accuracy variables are defined from Parts 2–4
results = {
    'Model': [
        'Baseline CNN',
        'ResNet50 (Fine-Tuned)',
        'VGG16 (Fine-Tuned)',
        'EfficientNetB0 (Fine-Tuned)',
        'MobileNetV2 (Fine-Tuned)',
    ],
    'Test Accuracy': [
        test_acc,
        resnet_ft_test_acc,
        vgg_test_acc,
        eff_test_acc,
        mob_test_acc,
    ]
}

df_results = pd.DataFrame(results)
df_results['Test Accuracy (%)'] = (df_results['Test Accuracy'] * 100).round(2)
df_results = df_results.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
df_results.index += 1  # rank starts from 1

print('=== Model Performance Summary (Ranked by Test Accuracy) ===')
print(df_results[['Model', 'Test Accuracy (%)']].to_string())


In [ ]:
# ── Bar Chart: Test Accuracy Comparison ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#2196F3', '#FF5722', '#9C27B0', '#4CAF50', '#F44336']
model_names = df_results['Model'].tolist()
accuracies  = df_results['Test Accuracy (%)'].tolist()

bars = ax.barh(model_names, accuracies, color=colors, edgecolor='black', linewidth=0.6)

# Annotate each bar with accuracy value
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height() / 2,
            f'{acc:.2f}%', va='center', ha='right', color='white',
            fontsize=11, fontweight='bold')

ax.set_xlabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — Test Accuracy on Held-Out Test Set',
             fontsize=13, fontweight='bold')
ax.set_xlim([0, 105])
ax.invert_yaxis()  # best model at top
plt.tight_layout()
plt.show()


## Per-Class F1-Score Comparison

Raw accuracy can be misleading in medical classification tasks. A model that is strong overall may still perform poorly on the most clinically critical class (COVID). Here we extract the per-class **F1-score** for every model — the harmonic mean of precision and recall — and plot them side by side.

In a COVID screening context, **COVID recall** is the most clinically important metric: a false negative (missed COVID case) carries far greater risk than a false positive.

In [ ]:
from sklearn.metrics import classification_report
import seaborn as sns

# ── Recompute predictions for all models ────────────────────────────────────
y_true = y_test_encoded
class_labels = list(label_encoder.classes_)

all_preds = {
    'Baseline CNN':              np.argmax(baseline_model.predict(X_test, verbose=0), axis=1),
    'ResNet50 (Fine-Tuned)':     np.argmax(resnet_model.predict(X_test, verbose=0),   axis=1),
    'VGG16 (Fine-Tuned)':        np.argmax(vgg_model.predict(X_test, verbose=0),      axis=1),
    'EfficientNetB0 (Fine-Tuned)': np.argmax(eff_model.predict(X_test, verbose=0),    axis=1),
    'MobileNetV2 (Fine-Tuned)':  np.argmax(mob_model.predict(X_test, verbose=0),      axis=1),
}

# ── Build per-class F1 DataFrame ────────────────────────────────────────────
f1_data = []
for model_name, y_pred in all_preds.items():
    report = classification_report(y_true, y_pred,
                                   target_names=class_labels,
                                   output_dict=True)
    for cls in class_labels:
        f1_data.append({
            'Model': model_name,
            'Class': cls,
            'F1-Score': round(report[cls]['f1-score'], 4),
            'Precision': round(report[cls]['precision'], 4),
            'Recall':    round(report[cls]['recall'], 4),
        })

df_f1 = pd.DataFrame(f1_data)

# ── Pivot and print ─────────────────────────────────────────────────────────
pivot_f1 = df_f1.pivot(index='Model', columns='Class', values='F1-Score')
print('=== Per-Class F1-Score Summary ===')
print(pivot_f1.to_string())


In [ ]:
# ── Grouped Bar Chart: F1-Score per Class per Model ────────────────────────
pivot_f1_plot = df_f1.pivot(index='Class', columns='Model', values='F1-Score')

ax = pivot_f1_plot.plot(
    kind='bar',
    figsize=(13, 6),
    colormap='tab10',
    edgecolor='black',
    linewidth=0.5,
    width=0.75
)
ax.set_title('Per-Class F1-Score — All Models', fontsize=13, fontweight='bold')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_ylim([0, 1.05])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.axhline(y=0.9, color='grey', linestyle='--', linewidth=0.8, label='0.90 threshold')
plt.tight_layout()
plt.show()


## Training & Validation Curves — All Models

Training and validation accuracy/loss curves reveal **how efficiently each model learns** and whether it tends to overfit. We plot all five models together on the same axes for direct visual comparison. The dashed vertical line (where applicable) marks the boundary between Phase 1 (feature extraction) and Phase 2 (fine-tuning).

In [ ]:
# ── Reconstruct full epoch sequences for each model ─────────────────────────
# Baseline CNN — single phase
histories = {
    'Baseline CNN': {
        'accuracy':     history_baseline.history['accuracy'],
        'val_accuracy': history_baseline.history['val_accuracy'],
        'loss':         history_baseline.history['loss'],
        'val_loss':     history_baseline.history['val_loss'],
    },
    'ResNet50 (FT)': {
        'accuracy':     history_resnet.history['accuracy']     + history_fine.history['accuracy'],
        'val_accuracy': history_resnet.history['val_accuracy'] + history_fine.history['val_accuracy'],
        'loss':         history_resnet.history['loss']         + history_fine.history['loss'],
        'val_loss':     history_resnet.history['val_loss']     + history_fine.history['val_loss'],
    },
    'VGG16 (FT)': {
        'accuracy':     acc_vgg,
        'val_accuracy': val_acc_vgg,
        'loss':         loss_vgg,
        'val_loss':     val_loss_vgg,
    },
    'EfficientNetB0 (FT)': {
        'accuracy':     acc_eff,
        'val_accuracy': val_acc_eff,
        'loss':         loss_eff,
        'val_loss':     val_loss_eff,
    },
    'MobileNetV2 (FT)': {
        'accuracy':     acc_mob,
        'val_accuracy': val_acc_mob,
        'loss':         loss_mob,
        'val_loss':     val_loss_mob,
    },
}

colors_map = {
    'Baseline CNN':        '#2196F3',
    'ResNet50 (FT)':       '#FF5722',
    'VGG16 (FT)':          '#9C27B0',
    'EfficientNetB0 (FT)': '#4CAF50',
    'MobileNetV2 (FT)':    '#F44336',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for model_name, hist in histories.items():
    c = colors_map[model_name]
    axes[0, 0].plot(hist['accuracy'],     color=c, label=model_name)
    axes[0, 1].plot(hist['val_accuracy'],  color=c, label=model_name)
    axes[1, 0].plot(hist['loss'],          color=c, label=model_name)
    axes[1, 1].plot(hist['val_loss'],      color=c, label=model_name)

# Fine-tuning boundary at epoch 10 for transfer learning models
for ax in axes.flat:
    ax.axvline(x=9, color='grey', linestyle=':', linewidth=1.2,
               label='Fine-Tune Start (epoch 10)')

titles = [
    ('Training Accuracy',   'Epoch', 'Accuracy'),
    ('Validation Accuracy', 'Epoch', 'Accuracy'),
    ('Training Loss',       'Epoch', 'Loss'),
    ('Validation Loss',     'Epoch', 'Loss'),
]
for ax, (title, xlabel, ylabel) in zip(axes.flat, titles):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Training & Validation Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Confusion Matrices — All Models

Confusion matrices provide a class-level breakdown of model errors. We plot all five models together so misclassification patterns can be compared directly. Pay particular attention to **COVID row** — missed COVID cases (false negatives) are the most clinically dangerous error type.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

cm_configs = [
    ('Baseline CNN',              y_pred_baseline,  'Blues'),
    ('ResNet50 (Fine-Tuned)',      y_pred_resnet,    'Oranges'),
    ('VGG16 (Fine-Tuned)',         y_pred_vgg,       'Purples'),
    ('EfficientNetB0 (Fine-Tuned)', y_pred_eff,      'Greens'),
    ('MobileNetV2 (Fine-Tuned)',   y_pred_mob,       'Reds'),
]

for ax, (name, preds, cmap) in zip(axes, cm_configs):
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_labels,
                yticklabels=class_labels,
                ax=ax, cbar=False)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('Confusion Matrices — All Five Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Hyperparameter & Training Configuration Summary

The table below documents the key hyperparameters and training choices for every model. Using consistent batch size, epoch count, and data splits across all models ensures that any performance differences reflect genuine architectural advantages, not differences in training protocol.

In [ ]:
hp_data = {
    'Model': [
        'Baseline CNN',
        'ResNet50 (Fine-Tuned)',
        'VGG16 (Fine-Tuned)',
        'EfficientNetB0 (Fine-Tuned)',
        'MobileNetV2 (Fine-Tuned)',
    ],
    'Optimizer (Ph1)':   ['Adam', 'Adam', 'Adam', 'Adam', 'Adam'],
    'LR (Ph1)':          ['1e-3', '1e-3', '1e-3', '1e-3', '1e-3'],
    'Optimizer (Ph2)':   ['—', 'Adam', 'Adam', 'Adam', 'Adam'],
    'LR (Ph2)':          ['—', '1e-5', '1e-5', '1e-5', '1e-5'],
    'Batch Size':        [32, 32, 32, 32, 32],
    'Epochs (Ph1+Ph2)':  ['10', '10+10', '10+10', '10+10', '10+10'],
    'Input Shape':       ['192×192×3'] * 5,
    'Loss Function':     ['Cat. CE'] * 5,
    'Fine-Tuned Layers': [
        'N/A (from scratch)',
        'Layers 143+ (last residual block)',
        'block5 (layers 15–18)',
        'Last 20 layers (MBConv blocks)',
        'Layers 100+ (upper inv. residuals)',
    ],
    'Dropout':           ['0.5'] * 5,
}

df_hp = pd.DataFrame(hp_data)
df_hp = df_hp.set_index('Model')
print('=== Hyperparameter & Training Configuration Summary ===')
print(df_hp.to_string())


## Best-Performing Model — Identification & Analysis

### Identifying the Best Model

The cell below programmatically identifies the best-performing model based on test accuracy, then prints a detailed classification report for it.

In [ ]:
# ── Identify best model programmatically ────────────────────────────────────
model_scores = {
    'Baseline CNN':                (test_acc,           np.argmax(baseline_model.predict(X_test, verbose=0), axis=1)),
    'ResNet50 (Fine-Tuned)':        (resnet_ft_test_acc, y_pred_resnet),
    'VGG16 (Fine-Tuned)':           (vgg_test_acc,       y_pred_vgg),
    'EfficientNetB0 (Fine-Tuned)':  (eff_test_acc,       y_pred_eff),
    'MobileNetV2 (Fine-Tuned)':     (mob_test_acc,       y_pred_mob),
}

best_name = max(model_scores, key=lambda k: model_scores[k][0])
best_acc, best_preds = model_scores[best_name]

print(f'>>> Best Model: {best_name}')
print(f'>>> Test Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)')
print()
print(f'=== {best_name} — Full Classification Report ===\n')
print(classification_report(y_true, best_preds, target_names=class_labels))


In [ ]:
# ── Confusion matrix of the best model ──────────────────────────────────────
cm_best = confusion_matrix(y_true, best_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title(f'Best Model: {best_name} — Confusion Matrix',
          fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


### Discussion: Why Does This Model Perform Best?

The performance comparison across five architectures reveals several key insights:

**1. Architecture–Data Fit**
The Baseline CNN, trained entirely from scratch on our balanced 4,032-image dataset, achieves strong performance because its three-block architecture (~128 filters max) is appropriately sized for 192×192 medical images. It has far fewer parameters than the pretrained architectures, giving it a natural regularisation advantage on a small dataset — it simply has less capacity to overfit.

**2. Transfer Learning: Domain Mismatch vs. Fine-Tuning Gain**
All four pretrained models (ResNet50, VGG16, EfficientNetB0, MobileNetV2) were originally trained on ImageNet — natural photographs with RGB colour distributions very different from greyscale-derived chest X-rays. This domain gap explains why frozen feature extraction underperforms in Phase 1. However, after fine-tuning (Phase 2), each model adapts its upper-layer representations to the X-ray domain, recovering most of the lost accuracy.

**3. Parameter Efficiency and Overfitting Risk**
- **VGG16** (~138 M parameters) carries the highest overfitting risk given our small training set, and its performance reflects this tension.
- **EfficientNetB0** (~5.3 M parameters) achieves an excellent balance: compound scaling ensures the network is neither too shallow nor too wide for our 192×192 inputs, and its MBConv blocks adapt more readily to the medical imaging domain during fine-tuning.
- **MobileNetV2** (~3.4 M parameters) is the most lightweight and shows robust generalisation, but its limited representational capacity may cap its ceiling accuracy.

**4. Clinical Metrics Beyond Accuracy**
Examining per-class F1-scores (Section 5.2) is equally important. A model with slightly lower overall accuracy but higher COVID recall may be preferable in a real screening setting, since missed COVID cases (false negatives) carry greater clinical risk than false alarms. The confusion matrices in Section 5.4 make these trade-offs explicit.

**5. Practical Recommendation**
- For **maximum accuracy** on this dataset: use the best-performing model identified above.
- For **deployment in resource-constrained settings** (mobile or edge devices): MobileNetV2 offers the best accuracy-to-efficiency trade-off.
- For **future work with larger datasets or domain-specific pretraining** (e.g., CheXNet pretrained on chest X-rays): EfficientNetB0 or ResNet50 would likely surpass the Baseline CNN as the domain mismatch disappears.

---

### Summary Table: All Models

| Rank | Model | Test Accuracy | Key Strength |
|:---:|---|:---:|---|
| 1 | *(Best — determined above)* | highest | |
| — | Baseline CNN | ~91–92% | Simplicity; no domain mismatch |
| — | EfficientNetB0 (FT) | competitive | Parameter-efficient; adapts well |
| — | MobileNetV2 (FT) | competitive | Lightweight; deployment-ready |
| — | VGG16 (FT) | competitive | Deep feature hierarchy |
| — | ResNet50 (FT) | ~88–90% | Residual learning; strong baseline |

*Exact rankings will reflect your training run results.*

# **6. Data Augmentation**

In this section, we retrain the best-performing model from Part 5 — **VGG16 (Fine-Tuned)** — using data augmentation.

The goal is to test whether augmentation improves generalization by exposing the model to more varied training examples. Because this is a medical imaging task, we use only mild augmentations that preserve the clinical structure of chest X-ray images.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mild augmentation only — keep medically plausible transformations
train_datagen_aug = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08,
    shear_range=0.05,
    horizontal_flip=False,   # safer for chest X-rays
    fill_mode='nearest'
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mild augmentation only — keep medically plausible transformations
train_datagen_aug = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08,
    shear_range=0.05,
    horizontal_flip=False,   # safer for chest X-rays
    fill_mode='nearest'
)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Rebuild VGG16 from scratch for the augmentation experiment
vgg_aug_base = VGG16(weights='imagenet', include_top=False, input_shape=(192, 192, 3))
vgg_aug_base.trainable = False

x = vgg_aug_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
vgg_aug_out = Dense(3, activation='softmax')(x)

vgg_aug_model = Model(inputs=vgg_aug_base.input, outputs=vgg_aug_out)

vgg_aug_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total layers in VGG16 base: {len(vgg_aug_base.layers)}')
print(f'Trainable variables (Phase 1): {len(vgg_aug_model.trainable_variables)}')

In [ ]:
batch_size = 32
epochs = 10

history_vgg_aug = vgg_aug_model.fit(
    train_datagen_aug.flow(X_train, y_train_cat, batch_size=batch_size),
    steps_per_epoch=len(X_train) // batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

vgg_aug_fe_loss, vgg_aug_fe_acc = vgg_aug_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'VGG16 + Augmentation (Feature Extraction) — Test Accuracy: {vgg_aug_fe_acc:.4f}')